# Experiment B: Bootstrap Confidence Intervals (§5.2)

2,000-replicate bootstrap analysis of accuracy, macro F1, and
cross-element error count for all correction variants.

## Setup

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

with open('../data/experiment_G_results.json', 'r') as f:
    boot = json.load(f)

print(f"Bootstrap replicates: {boot['n_bootstrap']}")
print(f"Test set size: {boot['n_test']}")

## Point Estimates

In [ ]:
print('Point estimates (test set):')
print('-' * 60)
for method, vals in boot['point_estimates'].items():
    print(f"{method:20s}  Acc={vals['accuracy']:.4f}  "
          f"F1={vals['macro_f1']:.4f}  CE={vals['cross_errors']}")

## Bootstrap 95% Confidence Intervals

In [ ]:
print('95% Bootstrap CIs:')
print('=' * 70)
for method, ci in boot['bootstrap_ci'].items():
    print(f"\n{method}:")
    print(f"  Accuracy: {ci['accuracy_mean']:.4f} "
          f"[{ci['accuracy_ci'][0]:.4f}, {ci['accuracy_ci'][1]:.4f}]")
    print(f"  Macro F1: {ci['macro_f1_mean']:.4f} "
          f"[{ci['macro_f1_ci'][0]:.4f}, {ci['macro_f1_ci'][1]:.4f}]")
    print(f"  Cross-elem errors: {ci['cross_err_mean']:.1f} "
          f"[{ci['cross_err_ci'][0]:.0f}, {ci['cross_err_ci'][1]:.0f}]")

## Delta vs Baseline (improvement CIs)

In [ ]:
print('Improvement over baseline (95% CI):')
print('=' * 70)
for method, d in boot['delta_vs_baseline'].items():
    print(f"\n{method}:")
    print(f"  ΔAcc CI: [{d['delta_acc_ci'][0]:+.4f}, {d['delta_acc_ci'][1]:+.4f}]  "
          f"P(better)={d['p_acc_better']:.3f}")
    print(f"  ΔF1  CI: [{d['delta_f1_ci'][0]:+.4f}, {d['delta_f1_ci'][1]:+.4f}]  "
          f"P(better)={d['p_f1_better']:.3f}")
    print(f"  ΔCE  CI: [{d['delta_ce_ci'][0]:+.0f}, {d['delta_ce_ci'][1]:+.0f}]  "
          f"P(better)={d['p_ce_better']:.3f}")

print('\n--- Key: Rule-based and BPC(constrained) CIs exclude zero for all metrics ---')

## Visualization

In [ ]:
methods = list(boot['bootstrap_ci'].keys())
acc_means = [boot['bootstrap_ci'][m]['accuracy_mean'] for m in methods]
acc_lo = [boot['bootstrap_ci'][m]['accuracy_ci'][0] for m in methods]
acc_hi = [boot['bootstrap_ci'][m]['accuracy_ci'][1] for m in methods]

fig, ax = plt.subplots(figsize=(8, 4))
x = range(len(methods))
yerr_lo = [m - lo for m, lo in zip(acc_means, acc_lo)]
yerr_hi = [hi - m for m, hi in zip(acc_means, acc_hi)]
ax.errorbar(x, acc_means, yerr=[yerr_lo, yerr_hi], fmt='o', capsize=5,
            markersize=8, color='#2196F3', ecolor='gray')
ax.set_xticks(x)
ax.set_xticklabels(methods, rotation=15, ha='right')
ax.set_ylabel('Accuracy')
ax.set_title('Bootstrap 95% CI – Accuracy by method')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/nb02_bootstrap_ci.png', dpi=150, bbox_inches='tight')
plt.show()

## Error Analysis Detail

In [ ]:
ea = boot['error_analysis']
print(f"Rule-based corrections: {ea['rule_corrected']} fixed, {ea['rule_worsened']} worsened")
print(f"Net improvement: {ea['rule_net']} samples")
print(f"\nConstrained vs Rule-based agreement: {ea['constrained_vs_rule_agree']}/"
      f"{ea['constrained_vs_rule_agree'] + ea['constrained_vs_rule_disagree']} "
      f"({ea['constrained_vs_rule_agree']/(ea['constrained_vs_rule_agree']+ea['constrained_vs_rule_disagree'])*100:.2f}%)")
print(f"\nCorrected transitions (cross→correct):")
for trans, count in ea['corrected_transitions'].items():
    print(f"  {trans}: {count}")

## Summary

- Rule-based and element-constrained variants: 95% CI for ΔAccuracy excludes zero → statistically significant.
- EN 206 soft prior: CI includes zero for accuracy → not significant alone, but reduces CE count (P=0.99).
- Element-constrained ≈ Rule-based: 8,419/8,420 agreement (99.99%), confirming mathematical equivalence (§3.7).